# Vault Explorer\n\nInteractive exploration of your knowledge base: catalog stats, wikilink graphs, citation networks, and reading suggestions.\n\nRun `kb ingest-all` and `kb backfill-references` before using this notebook to get the most out of it.

In [ ]:
import json\nimport re\nfrom collections import Counter\nfrom pathlib import Path\n\nimport frontmatter\nimport matplotlib.pyplot as plt\nimport networkx as nx\nimport pandas as pd\n\nROOT = Path(\".\").resolve()\nRAW = ROOT / \"raw\"\nWIKI = ROOT / \"wiki\"\n\nplt.rcParams.update({\"figure.dpi\": 120, \"figure.facecolor\": \"white\"})

## 1. Catalog overview

In [ ]:
catalog_path = RAW / \"_catalog.json\"\ncatalog = json.loads(catalog_path.read_text()) if catalog_path.exists() else []\n\nif not catalog:\n    print(\"Catalog is empty. Run `kb catalog` to register your papers.\")\nelse:\n    df_cat = pd.DataFrame(catalog)\n    print(f\"{len(df_cat)} papers in catalog\")\n    print(f\"  Extracted: {df_cat['extracted'].sum()}\")\n    print(f\"  Ingested:  {df_cat['ingested'].sum()}\")\n    print(f\"  Pending:   {(~df_cat['ingested']).sum()}\")\n    display(df_cat[[\"filename\", \"title\", \"year\", \"extracted\", \"ingested\"]].sort_values(\"year\"))

In [ ]:
if catalog:\n    all_kw = [kw for entry in catalog for kw in entry.get(\"keywords\", [])]\n    kw_counts = Counter(all_kw).most_common(20)\n    if kw_counts:\n        kws, counts = zip(*kw_counts)\n        fig, ax = plt.subplots(figsize=(8, 4))\n        ax.barh(range(len(kws)), counts, color=\"steelblue\")\n        ax.set_yticks(range(len(kws)))\n        ax.set_yticklabels(kws, fontsize=9)\n        ax.invert_yaxis()\n        ax.set_xlabel(\"Papers\")\n        ax.set_title(\"Most common keywords in catalog\")\n        plt.tight_layout()\n        plt.show()

## 2. Wiki article stats\n\nParse all summary frontmatter to get an overview of the wiki.

In [ ]:
def load_wiki_articles():\n    \"\"\"Load all wiki articles with their frontmatter and body.\"\"\"\n    articles = []\n    for subdir in [\"summaries\", \"concepts\", \"connections\"]:\n        folder = WIKI / subdir\n        if not folder.exists():\n            continue\n        for path in sorted(folder.glob(\"*.md\")):\n            if path.name.startswith(\"_\"):\n                continue\n            post = frontmatter.load(str(path))\n            wikilinks = re.findall(r\"\\[\\[([^\\]]+)\\]\\]\", post.content)\n            refs = post.get(\"references\", [])\n            articles.append({\n                \"file\": path.name,\n                \"stem\": path.stem,\n                \"type\": post.get(\"type\", subdir.rstrip(\"s\")),\n                \"title\": post.get(\"title\", path.stem),\n                \"sources\": post.get(\"sources\", []),\n                \"wikilinks\": wikilinks,\n                \"references\": refs,\n                \"words\": len(post.content.split()),\n            })\n    return articles\n\narticles = load_wiki_articles()\nprint(f\"{len(articles)} wiki articles loaded\")\n\nif articles:\n    df = pd.DataFrame(articles)\n    type_counts = df[\"type\"].value_counts()\n    print(f\"\\nBy type:\")\n    for t, n in type_counts.items():\n        print(f\"  {t}: {n}\")\n    print(f\"\\nTotal words: {df['words'].sum():,}\")\n    print(f\"Average wikilinks per article: {df['wikilinks'].apply(len).mean():.1f}\")\n    refs_with = df[\"references\"].apply(len)\n    print(f\"Articles with references: {(refs_with > 0).sum()} / {len(df)}\")\n    print(f\"Total references extracted: {refs_with.sum()}\")

## 3. Wikilink graph\n\nWhich articles link to which? Node size = number of incoming links (how central a concept is).

In [ ]:
if articles:\n    G_wiki = nx.DiGraph()\n    existing_stems = {a[\"stem\"] for a in articles}\n\n    for a in articles:\n        G_wiki.add_node(a[\"stem\"], type=a[\"type\"])\n        for link in set(a[\"wikilinks\"]):\n            G_wiki.add_edge(a[\"stem\"], link)\n\n    in_deg = dict(G_wiki.in_degree())\n    node_colors = []\n    palette = {\"summary\": \"#4a90d9\", \"concept\": \"#e8913a\", \"connection\": \"#50b87a\"}\n    for node in G_wiki.nodes():\n        ntype = G_wiki.nodes[node].get(\"type\", \"\")\n        if node not in existing_stems:\n            node_colors.append(\"#cccccc\")  # missing targets\n        else:\n            node_colors.append(palette.get(ntype, \"#999999\"))\n\n    node_sizes = [max(80, in_deg.get(n, 0) * 120) for n in G_wiki.nodes()]\n\n    fig, ax = plt.subplots(figsize=(12, 9))\n    pos = nx.spring_layout(G_wiki, k=1.8, iterations=60, seed=42)\n    nx.draw_networkx_edges(G_wiki, pos, alpha=0.15, arrows=True, arrowsize=8, ax=ax)\n    nx.draw_networkx_nodes(G_wiki, pos, node_size=node_sizes, node_color=node_colors, alpha=0.85, ax=ax)\n\n    # label only nodes with >= 2 incoming links\n    labels = {n: n for n in G_wiki.nodes() if in_deg.get(n, 0) >= 2}\n    nx.draw_networkx_labels(G_wiki, pos, labels, font_size=7, ax=ax)\n\n    from matplotlib.lines import Line2D\n    legend = [\n        Line2D([0], [0], marker=\"o\", color=\"w\", markerfacecolor=c, markersize=8, label=t)\n        for t, c in [*palette.items(), (\"missing\", \"#cccccc\")]\n    ]\n    ax.legend(handles=legend, loc=\"upper left\", fontsize=8)\n    ax.set_title(f\"Wikilink graph — {G_wiki.number_of_nodes()} nodes, {G_wiki.number_of_edges()} edges\")\n    ax.axis(\"off\")\n    plt.tight_layout()\n    plt.show()\n\n    # top linked-to targets\n    top_targets = sorted(in_deg.items(), key=lambda x: x[1], reverse=True)[:15]\n    print(\"Most linked-to nodes:\")\n    for name, deg in top_targets:\n        exists = \"exists\" if name in existing_stems else \"MISSING\"\n        print(f\"  {name}: {deg} incoming links ({exists})\")\nelse:\n    print(\"No wiki articles to graph.\")

## 4. Citation network\n\nBuilt from the `references:` frontmatter in each summary. Edges mean \"paper A cites paper B\" (both in the vault). Requires `kb backfill-references` to have run.

In [ ]:
def match_ref_to_catalog(ref, catalog_entries):\n    \"\"\"Match a reference dict to a catalog entry by first-author + year.\"\"\"\n    author = ref.get(\"author\", \"\").lower().strip()\n    year = ref.get(\"year\", 0)\n    if not author or not year:\n        return None\n    for e in catalog_entries:\n        if e.get(\"year\") == year and e.get(\"authors\") and e[\"authors\"][0].lower() == author:\n            return e\n    for e in catalog_entries:\n        if e.get(\"year\") == year and any(a.lower() == author for a in e.get(\"authors\", [])):\n            return e\n    return None\n\n# collect all references from summaries\nsummaries = [a for a in articles if a[\"type\"] == \"summary\"]\nall_refs = []\ncross_edges = []  # (citer_stem, cited_stem)\nexternal_refs = []  # refs not matched to catalog\n\nfor a in summaries:\n    for ref in a[\"references\"]:\n        if not isinstance(ref, dict):\n            continue\n        match = match_ref_to_catalog(ref, catalog)\n        if match:\n            cited_stem = match[\"filename\"].rsplit(\".\", 1)[0]\n            if cited_stem != a[\"stem\"]:\n                cross_edges.append((a[\"stem\"], cited_stem))\n        else:\n            author = ref.get(\"author\", \"\").strip()\n            year = ref.get(\"year\", 0)\n            title = ref.get(\"title\", \"\").strip()\n            if author and year:\n                external_refs.append((author.lower(), year, title, a[\"stem\"]))\n        all_refs.append(ref)\n\ntotal_refs = sum(len(a[\"references\"]) for a in summaries)\nprint(f\"{len(summaries)} summaries, {total_refs} total references extracted\")\nprint(f\"{len(cross_edges)} cross-citations between vault papers\")\nprint(f\"{len(external_refs)} references to external papers\")\n\nif not total_refs:\n    print(\"\\nNo references found. Run `kb backfill-references` to extract them.\")

In [ ]:
if cross_edges:\n    G_cite = nx.DiGraph()\n    for citer, cited in cross_edges:\n        G_cite.add_edge(citer, cited)\n\n    in_deg = dict(G_cite.in_degree())\n    node_sizes = [max(200, in_deg.get(n, 0) * 300) for n in G_cite.nodes()]\n\n    fig, ax = plt.subplots(figsize=(10, 8))\n    pos = nx.spring_layout(G_cite, k=2.0, iterations=50, seed=42)\n    nx.draw_networkx_edges(G_cite, pos, alpha=0.3, arrows=True,\n                           arrowsize=12, edge_color=\"#666\", ax=ax)\n    nx.draw_networkx_nodes(G_cite, pos, node_size=node_sizes,\n                           node_color=\"#4a90d9\", alpha=0.8, ax=ax)\n    nx.draw_networkx_labels(G_cite, pos, font_size=7, ax=ax)\n    ax.set_title(f\"Citation network — {G_cite.number_of_nodes()} papers, {G_cite.number_of_edges()} citations\")\n    ax.axis(\"off\")\n    plt.tight_layout()\n    plt.show()\n\n    print(\"\\nMost cited within vault:\")\n    for stem, deg in sorted(in_deg.items(), key=lambda x: x[1], reverse=True)[:10]:\n        print(f\"  {stem}: cited by {deg} vault papers\")\nelif total_refs:\n    print(\"No cross-citations found between vault papers (all references are to external works).\")\nelse:\n    print(\"Skipping citation graph — no references extracted yet.\")

## 5. Reading suggestions\n\nPapers cited by multiple vault sources but not yet in your catalog. These are your highest-value next reads.

In [ ]:
if external_refs:\n    # group by (author, year), keep best title, collect citers\n    grouped = {}\n    best_title = {}\n    for author, year, title, citer in external_refs:\n        key = (author, year)\n        grouped.setdefault(key, set()).add(citer)\n        if title and len(title) > len(best_title.get(key, \"\")):\n            best_title[key] = title\n\n    ranked = sorted(grouped.items(), key=lambda x: len(x[1]), reverse=True)\n    ranked = [(k, citers) for k, citers in ranked if len(citers) >= 2]\n\n    if ranked:\n        rows = []\n        for (author, year), citers in ranked[:25]:\n            rows.append({\n                \"Author\": author.title(),\n                \"Year\": year,\n                \"Title\": best_title.get((author, year), \"\")[:60],\n                \"Cited by\": len(citers),\n                \"Citing papers\": \", \".join(sorted(citers)),\n            })\n        df_suggest = pd.DataFrame(rows)\n        print(f\"{len(ranked)} external papers cited by 2+ vault sources:\\n\")\n        display(df_suggest)\n    else:\n        print(\"No external papers are cited by 2+ vault sources yet.\")\nelse:\n    print(\"No external references to analyze. Run `kb backfill-references` first.\")

## 6. Publication timeline\n\nWhen were your vault papers published? Helps spot temporal gaps in coverage.

In [ ]:
if catalog:\n    years = [e[\"year\"] for e in catalog if e.get(\"year\")]\n    if years:\n        year_counts = Counter(years)\n        yr_range = range(min(years), max(years) + 1)\n        counts = [year_counts.get(y, 0) for y in yr_range]\n\n        fig, ax = plt.subplots(figsize=(10, 3.5))\n        ax.bar(yr_range, counts, color=\"steelblue\", width=0.8)\n        ax.set_xlabel(\"Year\")\n        ax.set_ylabel(\"Papers\")\n        ax.set_title(\"Publication years of vault papers\")\n        ax.set_xticks([y for y in yr_range if y % 5 == 0 or y == min(years) or y == max(years)])\n        plt.tight_layout()\n        plt.show()\n\n        decade_gaps = [y for y in yr_range if year_counts.get(y, 0) == 0]\n        if decade_gaps:\n            print(f\"Years with no papers: {', '.join(map(str, decade_gaps))}\")\nelse:\n    print(\"No catalog data.\")

## 7. Concept co-occurrence\n\nWhich concepts tend to appear together in the same summaries? Thicker edges = more papers mentioning both.

In [ ]:
if summaries:\n    from itertools import combinations\n\n    cooccur = Counter()\n    concept_freq = Counter()\n    for a in summaries:\n        concepts = sorted(set(a[\"wikilinks\"]))\n        for c in concepts:\n            concept_freq[c] += 1\n        for pair in combinations(concepts, 2):\n            cooccur[pair] += 1\n\n    # filter to pairs appearing in 2+ papers\n    strong = {pair: n for pair, n in cooccur.items() if n >= 2}\n\n    if strong:\n        G_co = nx.Graph()\n        for (a, b), weight in strong.items():\n            G_co.add_edge(a, b, weight=weight)\n\n        freq = {n: concept_freq[n] for n in G_co.nodes()}\n        node_sizes = [max(100, freq.get(n, 1) * 80) for n in G_co.nodes()]\n        edge_widths = [G_co[u][v][\"weight\"] * 1.5 for u, v in G_co.edges()]\n\n        fig, ax = plt.subplots(figsize=(12, 9))\n        pos = nx.spring_layout(G_co, k=1.5, iterations=50, seed=42)\n        nx.draw_networkx_edges(G_co, pos, width=edge_widths, alpha=0.25, ax=ax)\n        nx.draw_networkx_nodes(G_co, pos, node_size=node_sizes,\n                               node_color=\"#e8913a\", alpha=0.8, ax=ax)\n        nx.draw_networkx_labels(G_co, pos, font_size=7, ax=ax)\n        ax.set_title(f\"Concept co-occurrence (pairs in 2+ papers) — {G_co.number_of_nodes()} concepts\")\n        ax.axis(\"off\")\n        plt.tight_layout()\n        plt.show()\n    else:\n        print(\"Not enough concept co-occurrence yet (need the same pair in 2+ papers).\")\n\n    # top concepts by frequency\n    print(\"\\nMost referenced concepts:\")\n    for concept, freq in concept_freq.most_common(15):\n        print(f\"  {concept}: appears in {freq} summaries\")\nelse:\n    print(\"No summaries to analyze.\")

## 8. Coverage gaps\n\nConcepts mentioned in summaries but with no dedicated article in `wiki/concepts/`.

In [ ]:
if articles:\n    all_stems = {a[\"stem\"] for a in articles}\n    all_links = Counter()\n    for a in articles:\n        for link in a[\"wikilinks\"]:\n            all_links[link] += 1\n\n    missing = {link: count for link, count in all_links.items() if link not in all_stems}\n    missing_sorted = sorted(missing.items(), key=lambda x: x[1], reverse=True)\n\n    if missing_sorted:\n        print(f\"{len(missing_sorted)} wikilink targets have no article yet:\\n\")\n        rows = [{\"Concept\": name, \"Referenced by\": count} for name, count in missing_sorted[:30]]\n        display(pd.DataFrame(rows))\n        print(f\"\\nRun `kb compile-concepts` to auto-generate articles for the most common ones.\")\n    else:\n        print(\"All wikilink targets have corresponding articles.\")\nelse:\n    print(\"No articles to check.\")